### Data Ingestion


In [ ]:
from langchain_core.documents import Document


In [ ]:
doc = Document(
    page_content = "This is main text context i am using to RAG",
    metadata = {
        "source":"example.txt",
        "pages":1,
        "author":"Muhammad waqas",
        "date_created":"2026-08-05"
    }
)
doc

In [ ]:
import os 

os.makedirs("../data/text_file", exist_ok = True)

In [ ]:
sample_texts = {"../data/text_file/python_intro.txt":"""Python ek high-level, interpreted, general-purpose programming language hai. Iska syntax simple aur readable hota hai, isi wajah se beginners aur professional developers dono use karte hain.

Tum AI/ML aur RAG seekh rahe ho, isliye Python ki fundamentals strong karna important hai.
1. Python kya hai?
Python se hum different types ke software aur applications bana sakte hain:
- AI & Machine Learning: Scikit-learn, PyTorch, TensorFlow
- Data Science: Pandas, NumPy, Matplotlib
- Backend Development: FastAPI, Django, Flask
- Generative AI: LangChain, LlamaIndex
- Automation: Files, Excel, web automation""",

    "../data/text_file/machine_learning_intro.txt":""""Machine Learning (ML) – Introduction
Machine Learning, Artificial Intelligence (AI) ki ek important branch hai. Ismein hum computer ko data se patterns seekhna sikhate hain, taake woh naye data par predictions ya decisions kar sake.

Simple words mein:

Traditional Programming: Hum computer ko rules dete hain aur woh un rules ke mutabiq output deta hai.

Machine Learning: Hum computer ko data aur correct examples dete hain, aur algorithm unmein patterns learn karta hai.
1. AI vs ML vs DL
In teen terms ko samajhna bohat important hai.
Artificial Intelligence (AI)

Machines performing intelligent tasks

Machine Learning (ML)

Learning patterns from data

Deep Learning (DL)

Learning using neural networks


Deep Learning, Machine Learning ka subset hai aur Machine Learning, AI ka subset hai."""


}

for filepath,content in sample_texts.items():
    with open(filepath, 'w',encoding='utf-8')as f:
        f.write(content)


print('Sapmle texts file created!')

In [ ]:
### Text Loader

from langchain_community.document_loaders import TextLoader

loader = TextLoader('../data/text_file/machine_learning_intro.txt', encoding = "utf-8")
document = loader.load()
print(document)

In [ ]:
### Directory loader
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/text_file",
    glob="*.txt", 
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'},
    show_progress=True
)


documents = dir_loader.load()
print('Total document:',len(documents))

In [ ]:
### Directory loader pdf
from langchain_community.document_loaders import PyPDFLoader , PyMuPDFLoader, DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="*.pdf", 
    loader_cls=PyMuPDFLoader,
    show_progress=True
)


pdf_document = dir_loader.load()
pdf_document

In [ ]:
print("Total PDF pages:", len(pdf_document))

print("\nFirst page text:")
print(pdf_document[0].page_content[:500])

print("\nMetadata:")
print(pdf_document[0].metadata)

### Embeding And VectorstoreDB

In [ ]:
import numpy as np
import chromadb
from rag.vector_store import stable_chunk_id
from typing import List,Dict,Any 
from sentence_transformers import SentenceTransformer
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
class EmbeddingManager:
    def __init__(self,model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model() 

    def _load_model(self):
        try:
            print(f"Load Embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded Successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}") 
        except Exception as e:
            print(f"Error loaded model: {self.model_name}:{e}")
            raise 

    def _generate_embaddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Modelnot loaded") 

        print(f"Generation Embedding for {len(texts)} texts...")
        embeddings = self.model.encode(
            texts,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
        print(f"Generated embedding with shape: {embeddings.shape}") 
        return embeddings         
    


embedding_manager = EmbeddingManager()
embedding_manager

### Vector Store


In [ ]:
class VectorStore:
    def __init__(self,collection_name: str = "pdf_chunks", persist_directory: str = "../data/Vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path= self.persist_directory)


            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF Document Embedding for RAG"}
            )
            print(f"Vector store initialize. collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        
        except Exception as e:
            print(f"Error Initializing Vector Store: {e}")
            raise         



    def add_documents(self, documents: List[Any], embeddings: np.ndarray):

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        unique_documents = {}
        for doc, embedding_vector in zip(documents, embeddings):
            unique_documents.setdefault(doc.page_content, (doc, embedding_vector))

        print(f"Syncing {len(unique_documents)} unique documents to vector store...")

        #Prepare the data of chromadb
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embeddings_vector) in enumerate(unique_documents.values()):
            doc_id = stable_chunk_id(doc)
            ids.append(doc_id)

            #prepear metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            #documents content
            documents_text.append(doc.page_content)

            #Embeddings
            embeddings_list.append([float(value) for value in embeddings_vector])

        #Add to collection
        try:
            existing = self.collection.get(include=["documents"])
            new_content = set(documents_text)
            replaced_ids = [
                old_id
                for old_id, old_text in zip(existing["ids"], existing["documents"])
                if old_text in new_content and old_id not in ids
            ]
            if replaced_ids:
                self.collection.delete(ids=replaced_ids)

            self.collection.upsert(
                ids = ids,
                embeddings=embeddings_list,
                metadatas = metadatas,
                documents=documents_text
            )

            print(f"Successfully synced {len(ids)} unique documents to vector store") 
            print(f"Total documents in collection: {self.collection.count()}") 

        except Exception as e:
            print(f"Error adding document  to vector store: {e}")
            raise


vectorstore = VectorStore()
vectorstore               

        

In [ ]:
texts = [doc.page_content for doc in pdf_document]

embeddings = embedding_manager._generate_embaddings(texts)

In [ ]:
print(embeddings.shape)

In [ ]:
vectorstore.add_documents(pdf_document, embeddings)

In [ ]:
print("Total Chunks:", vectorstore.collection.count())

In [ ]:
all_chunks = vectorstore.collection.get(
    include=["documents", "metadatas"]
)

for i, chunk in enumerate(all_chunks["documents"][:3]):

    print(f"\n{'='*70}")
    print(f"CHUNK {i+1}")
    print(f"{'='*70}")

    print(chunk)

### Spliting into Chunks


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

In [ ]:
chunks = text_splitter.split_documents(pdf_document)

In [ ]:
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- CHUNK {i+1} ---")
    print(chunk.page_content)

In [ ]:
texts = [chunk.page_content for chunk in chunks]

chunk_embeddings = embedding_manager._generate_embaddings(texts)

In [ ]:
chunk_vectorstore = VectorStore(
    collection_name="pdf_chunks"
)

In [ ]:
chunk_vectorstore.add_documents(
    chunks,
    chunk_embeddings
)

In [ ]:
print(chunk_vectorstore.collection.count())

In [ ]:
print(f"Total chunks: {len(chunks)}")

In [ ]:
print(chunk_vectorstore.collection.name)
print(chunk_vectorstore.collection.count())

In [ ]:
print(vectorstore.collection.name)
print(vectorstore.collection.count())

In [ ]:
all_chunks = chunk_vectorstore.collection.get(
    include=["documents", "metadatas"]
)

print("Total:", len(all_chunks["ids"]))

print(all_chunks["ids"][:5])
print(all_chunks["ids"][100:105])

In [ ]:
count_before = chunk_vectorstore.collection.count()
chunk_vectorstore.add_documents(chunks, chunk_embeddings)
assert chunk_vectorstore.collection.count() == count_before
print(f"Safe re-index count: {count_before}")

### Retriever Pipeline From VectorStore

In [ ]:
class RAGRetriever:
    def __init__(self,vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int=5, score_threshold: float=0.0)-> List[Dict[str, Any]]:
        if top_k <= 0:
            raise ValueError("top_k must be greater than zero")
        if self.vector_store.collection.count() == 0:
            return []

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, score_threshold: {score_threshold}")    

        query_embedding = self.embedding_manager._generate_embaddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=min(top_k, self.vector_store.collection.count())
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance / 2

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })  
            print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            return retrieved_docs
                    

        except Exception as e :
            print(f"Error during retrieval: {e}")
            raise

rag_retriever = RAGRetriever(chunk_vectorstore, embedding_manager)
rag_retriever        

In [ ]:
results = rag_retriever.retrieve(
    query="What is Artificial Intelligence?",
    top_k=5
)

In [ ]:
for doc in results:
    print("\n", "="*60)
    print("Rank:", doc["rank"])
    print("Distance:", doc["distance"])
    print("Content:", doc["content"])

### Integration VectorDB context pipeline with LLM Output

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", """
    You are an AI assistant.
    Answer the user's question clearly and accurately.
    Use only the provided context.
    If the answer is not available in context, say you don't know.
    """),

    ("human", """
    Context:
    {context}

    Question:
    {query}

    Answer:
    """)
])

In [ ]:
formatted_prompt = prompt.format_messages(
    context="Artificial Intelligence is a field of computer science.",
    query="What is Artificial Intelligence?"
)

print(formatted_prompt)

In [ ]:
from langchain_core.runnables import RunnablePassthrough

chain = RunnablePassthrough()

result = chain.invoke("What is Artificial Intelligence?")

print(result)

In [ ]:
from langchain_core.runnables import RunnablePassthrough

chain = RunnablePassthrough.assign(
    course=lambda x: "Artificial Intelligence"
)

result = chain.invoke({
    "name": "Waqas",
    "subject": "Python"
})

print(result)

In [ ]:
rag_chain = RunnablePassthrough.assign(
    context=lambda x: "\n\n".join(
        doc["content"]
        for doc in rag_retriever.retrieve(x["query"], top_k=3)
    )
)

result = rag_chain.invoke({
    "query": "What is Artificial Intelligence?"
})

print(result["query"])
print(result["context"])

In [ ]:
from langchain_core.runnables import RunnableLambda

def upper_text(text):
    return text.upper()

chain = RunnableLambda(upper_text)

result = chain.invoke("hello artificial intelligence")

print(result)

In [ ]:
def format_docs(docs):
    return "\n\n".join(
        doc["content"] for doc in docs
    )



format_chain = RunnableLambda(format_docs)

In [ ]:
results = rag_retriever.retrieve(
    "What is Artificial Intelligence?",
    top_k=3
)

context = format_chain.invoke(results)

print(context)

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

chain = RunnableParallel({
    "query": RunnablePassthrough(),
    "course": RunnableLambda(lambda x: "Artificial Intelligence"),
    "status": RunnableLambda(lambda x: "Learning RAG")
})

result = chain.invoke("What is AI?")

print(result)

In [ ]:
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda
)

# 1. Query length calculate karne ka function
def calculate_length(query):
    return len(query)


# 2. Parallel Chain
chain = RunnableParallel({

    "query": RunnablePassthrough(),

    "course": RunnableLambda(
        lambda x: "Artificial Intelligence"
    ),

    "status": RunnableLambda(
        lambda x: "Learning RAG"
    ),

    "query_length": RunnableLambda(
        calculate_length
    )

})


# 3. Invoke Chain
result = chain.invoke("What is AI?")

# 4. Print Result
print(result)

In [ ]:
from langchain_core.runnables import RunnableLambda

def convert_upper(text):
    return text.upper()

def add_message(text):
    return f"Your Query: {text}"

chain = (
    RunnableLambda(convert_upper)
    | RunnableLambda(add_message)
)

result = chain.invoke("What is AI?")

print(result)

In [ ]:
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda
)

# Step 1: Query Processing
def process_query(query):
    return query.strip().capitalize()


# Step 2: Query Analysis
def get_length(query):
    return len(query)


def get_words(query):
    return query.split()


# Step 3: Parallel Processing
parallel_chain = RunnableParallel({

    "original_query": RunnablePassthrough(),

    "query_length": RunnableLambda(get_length),

    "words": RunnableLambda(get_words),

    "course": RunnableLambda(
        lambda x: "Artificial Intelligence"
    )
})


# Step 4: Final Output
def format_output(data):
    return {
        "message": "RAG Query Processed Successfully",
        "data": data
    }


# Step 5: Complete Sequence
chain = (
    RunnableLambda(process_query)
    | parallel_chain
    | RunnableLambda(format_output)
)


# Step 6: Execute
result = chain.invoke("  what is artificial intelligence?  ")

print(result)

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""
    You are an AI teacher.

    Explain the following topic in simple words:

    Topic: {topic}

    Give:
    1. Definition
    2. Examples
    3. Applications
    """,
    input_variables=["topic"]
)

result = prompt.invoke({
    "topic": "Artificial Intelligence"
})

print(result)
print(result.to_string())

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert AI teacher."),
    ("human", "Explain {topic} in simple words with examples.")
])

result = chat_prompt.invoke({
    "topic": "Machine Learning"
})

print(result)
print(result.to_messages())

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Python teacher."),
    ("human", "Explain {topic} with {examples} examples.")
])

result = prompt.invoke({
    "topic": "Python Functions",
    "examples": 3
})

print(result.to_string())

In [ ]:
### Run the structured RAG package with ChatPromptTemplate
from rag.config import load_settings
from rag.embeddings import SentenceTransformerEmbeddings
from rag.indexing import index_corpus
from rag.vector_store import ChromaVectorStore

settings = load_settings()
embedding_model = SentenceTransformerEmbeddings(settings.embedding_model_name)
vector_store = ChromaVectorStore(
    settings.vector_store_dir,
    collection_name=settings.collection_name,
)

# Sync on each run: deterministic IDs prevent duplicates and stale chunks are removed.
indexed_count = index_corpus(settings, vector_store, embedding_model)
print(f"Synced {indexed_count} unique chunks.")

In [ ]:
### Ask a question with the ChatPromptTemplate RAG chain
from rag.chain import create_rag_chain
from rag.config import create_groq_chat_model

llm = create_groq_chat_model(settings)
rag_chain = create_rag_chain(vector_store, embedding_model, llm, top_k=3)
rag_result = rag_chain.invoke({"query": "What is Artificial Intelligence?"})

print("Answer:\n", rag_result["answer"])
print("\nSources:")
for source in rag_result["sources"]:
    page = f", page {source['page']}" if source["page"] is not None else ""
    print(f"- {source['source']}{page}")